# Hands-On 1 · Gradient Descent by Hand

**Introduction to Machine Learning** · MDMC Master, Trieste · Day 1, *Learning from Data* · Monday 9 November 2026 · Paolo Muratore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/myscience/intro-ml-mdmc-2026/blob/main/notebooks/N1_gradient_descent.ipynb)

You will fit a straight line by gradient descent, writing every piece yourself: the loss, its gradient and the update loop, and then you will see scikit-learn do the same job in one line. It is the recipe from this morning's slides *The Loss*, *Gradient Descent*, *What If the Step Is Too Big?* and *Linear Regression*, put to work.

| Tier | For whom | Time |
|---|---|---|
| ★ Core | everyone | about 35 min |
| ★★ Stretch | when you finish the Core | about 20 min |
| ★★★ Challenge | when you want more | open-ended |

**How it works.** Fill in every `____` and every `# YOUR CODE HERE` followed by `...`, then run the cell. Each exercise ends with a **checkpoint** that tells you whether you got it right, and a **Question** to answer in a sentence or two.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression

SEED = 0
rng = np.random.default_rng(SEED)

# Course palette (the colours of the slides, light version)
YELLOW, CYAN, MAGENTA, LIME = "#D99E00", "#0094AE", "#D1006C", "#7E8E00"
INK, GREY = "#111111", "#6B6B6B"
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[YELLOW, CYAN, MAGENTA, LIME]),
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.figsize": (6, 4), "font.size": 11,
})
print("Ready.")

## ★ Core: gradient descent for a straight line

The recipe from the slide *The Recipe*: **data** (a line plus noise), a **model** $\hat y = w\,x + b$, a **loss** (the mean squared error) and an **optimiser** (gradient descent). The data are given; you write the other three.

### Exercise 1 · NumPy warm-up (3 min)

NumPy applies arithmetic to every entry of an array at once, with no `for` loop. That is all you need for the loss and its gradient. Fill in the three blanks: an operator, a function name, and another operator.

In [ ]:
a = np.array([1.0, 2.0, 3.0, 4.0])
c = np.array([2.0, 0.0, -1.0, 1.0])

d = a - c  # entry by entry: [-1, 2, 4, 3]
d_squared = d ____ 2  # the square of every entry
avg = np.____(d_squared)  # the average of the entries
avg_product = np.mean(a ____ c)  # the average of the products a_i * c_i

print("d_squared   =", d_squared)
print("avg         =", avg)
print("avg_product =", avg_product)

In [ ]:
# Checkpoint
assert np.allclose(d_squared, [1, 4, 16, 9]), "d_squared should hold the square of each entry of d"
assert np.isclose(avg, 7.5), "avg should be the mean of d_squared"
assert np.isclose(avg_product, 0.75), "avg_product should be the mean of a * c, entry by entry"
print("Well done. avg is a mean squared error: the average squared difference between a and c.")

**Question.** For `d = np.array([-3.0, 3.0])`, `np.mean(d)` is 0, although both entries are far from zero. What is `np.mean(d ** 2)`? Why is it the better measure of "how far"?

*Your answer:* 

### The data: a line plus noise

Just run this cell. As in the lecture, the data come from a true line $y = w_\text{true}\,x + b_\text{true}$, plus noise. In real life nobody tells you `w_true` and `b_true`; here we know them, so we can check whether learning finds them.

In [ ]:
N = 100
w_true, b_true = 2.0, 1.0

data_rng = np.random.default_rng(SEED)    # its own generator: re-running this cell gives the same data
x = data_rng.normal(0, 1, size=N)         # the inputs
noise = data_rng.normal(0, 1, size=N)     # measurement noise
y = w_true * x + b_true + noise           # the answers: the true line, plus noise

fig, ax = plt.subplots()
ax.scatter(x, y, s=20, color=INK, alpha=0.7)
ax.set(xlabel="$x$", ylabel="$y$", title=f"{N} noisy points")
plt.show()

### Exercise 2 · The model and the loss (6 min)

The model is a straight line with two knobs, $\hat y = w\,x + b$. The loss is the mean squared error from the slide *The Loss*:

$$L(w, b) = \frac{1}{N}\sum_{i=1}^{N} r_i^2, \qquad r_i = y_i - \hat y_i .$$

Write the body of each function: one line each. The inputs are NumPy arrays, so no loop is needed.

In [ ]:
def predict(x, w, b):
    """The model: the line w * x + b, evaluated at every entry of x."""
    # YOUR CODE HERE
    ...


def mse(y, y_hat):
    """The loss: the mean of the squared residuals y - y_hat."""
    # YOUR CODE HERE
    # HINT: you did the same in Exercise 1, with a and c
    ...

In [ ]:
# Checkpoint
assert predict(np.zeros(3), 1.0, 1.0) is not None, "predict returns nothing: did you write `return ...`?"
assert mse(np.zeros(3), np.ones(3)) is not None, "mse returns nothing: did you write `return ...`?"
assert np.allclose(predict(np.array([0.0, 1.0, 2.0]), w=3.0, b=1.0), [1, 4, 7]), \
    "predict(x=[0, 1, 2], w=3, b=1) should give [1, 4, 7]"
assert np.isclose(mse(np.array([1.0, 2.0, 3.0]), np.array([1.0, 2.0, 5.0])), 4 / 3), \
    "mse should average the squared residuals: here (0 + 0 + 4) / 3"
print("Both functions work.")

In [ ]:
# Three settings of the knobs, as on the slide "Which Line Is Best?". Run to see how wrong each one is.
candidates = {"A": (0.5, 3.0), "B": (2.0, 1.0), "C": (3.5, -1.0)}

xs = np.linspace(x.min(), x.max(), 2)
fig, ax = plt.subplots()
ax.scatter(x, y, s=20, color=INK, alpha=0.7)
for (name, (w_c, b_c)), color in zip(candidates.items(), [CYAN, YELLOW, MAGENTA]):
    loss = mse(y, predict(x, w_c, b_c))
    ax.plot(xs, predict(xs, w_c, b_c), color=color, lw=2,
            label=f"{name}:  w = {w_c},  b = {b_c},  loss = {loss:.2f}")
ax.set(xlabel="$x$", ylabel="$y$", title="Which line is best?")
ax.legend(fontsize=9)
plt.show()

**Question.** Line B is the true line, $w = 2$ and $b = 1$. Why is its loss not zero? Could any straight line reach a loss of zero on these points?

*Your answer:* 

### Exercise 3 · The gradient (8 min)

Gradient descent needs the slope of the loss along each knob. Differentiating $L$ (the chain rule on each $r_i^2$) gives

$$\frac{\partial L}{\partial w} = -\frac{2}{N}\sum_{i=1}^{N} x_i\, r_i, \qquad \frac{\partial L}{\partial b} = -\frac{2}{N}\sum_{i=1}^{N} r_i .$$

Translate the two formulas into NumPy, as `dL_dw` and `dL_db`. The residuals `r` are already computed for you. Remember that $\frac{1}{N}\sum_i$ is `np.mean`.

In [ ]:
def gradients(x, y, w, b):
    """The two partial derivatives (dL/dw, dL/db) of the MSE, at the knobs (w, b)."""
    r = y - predict(x, w, b)    # the residuals r_i = y_i - y_hat_i
    # YOUR CODE HERE
    ...
    return dL_dw, dL_db

In [ ]:
# Checkpoint: compare with a numerical estimate of the slope.
# Nudge one knob by a tiny h and see how much the loss changes: (L(w + h) - L(w - h)) / 2h.
h = 1e-5
w0, b0 = 0.5, -0.3
num_dw = (mse(y, predict(x, w0 + h, b0)) - mse(y, predict(x, w0 - h, b0))) / (2 * h)
num_db = (mse(y, predict(x, w0, b0 + h)) - mse(y, predict(x, w0, b0 - h))) / (2 * h)
dL_dw, dL_db = gradients(x, y, w0, b0)
assert np.isclose(dL_dw, num_dw, rtol=1e-4), \
    f"Your dL/dw is {dL_dw:.4f}, but nudging w says {num_dw:.4f}. Check the sign, the factor 2 and the x_i."
assert np.isclose(dL_db, num_db, rtol=1e-4), \
    f"Your dL/db is {dL_db:.4f}, but nudging b says {num_db:.4f}. Check the sign and the factor 2."
print("Your gradient matches the numerical slope.")
print("At w = 0, b = 0:  dL/dw = {:.3f},  dL/db = {:.3f}".format(*gradients(x, y, 0.0, 0.0)))

**Question.** At $w = 0$, $b = 0$ both derivatives are negative. Will the first step of gradient descent make $w$ and $b$ larger or smaller? Does that make sense, looking at the data?

*Your answer:* 

### Exercise 4 · Rolling downhill (8 min)

Now the optimiser, from the slide *Gradient Descent*: start somewhere, look at the slope under your feet, take a small step downhill, repeat.

$$w \leftarrow w - \eta\,\frac{\partial L}{\partial w}, \qquad b \leftarrow b - \eta\,\frac{\partial L}{\partial b}$$

The learning rate $\eta$ is `eta` in the code. Write the two update lines inside the loop.

In [ ]:
def fit_line(x, y, eta, n_steps, w=0.0, b=0.0):
    """Gradient descent on the MSE. Returns the final knobs and the loss at every step."""
    losses = []
    for step in range(n_steps):
        losses.append(mse(y, predict(x, w, b)))   # how wrong we are now
        dL_dw, dL_db = gradients(x, y, w, b)      # the slope under our feet
        # YOUR CODE HERE
        ...
    return w, b, losses


w, b, losses = fit_line(x, y, eta=0.1, n_steps=50)
print(f"learned:  w = {w:.3f},  b = {b:.3f}")
print(f"true:     w = {w_true:.3f},  b = {b_true:.3f}")

In [ ]:
# Checkpoint
assert losses[-1] < losses[0], "The loss did not go down. Do you step against the gradient (with a minus sign)?"
assert abs(w - w_true) < 0.2 and abs(b - b_true) < 0.2, \
    f"w = {w:.2f} and b = {b:.2f} are still far from the true values (2, 1). Check the two update lines."
print("Gradient descent found the line.")
print(f"loss of the learned line: {mse(y, predict(x, w, b)):.4f}")
print(f"loss of the true line:    {mse(y, predict(x, w_true, b_true)):.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(losses, color=MAGENTA, lw=2)
ax1.set(xlabel="step", ylabel="loss $L(w, b)$", title="The loss goes down")
ax2.scatter(x, y, s=20, color=INK, alpha=0.7, label="data")
ax2.plot(xs, predict(xs, w_true, b_true), color=GREY, lw=2, ls="--", label="true line")
ax2.plot(xs, predict(xs, w, b), color=YELLOW, lw=2, label="learned line")
ax2.set(xlabel="$x$", ylabel="$y$", title="The learned line")
ax2.legend()
plt.show()

**Question.** Your $w$ and $b$ are close to 2 and 1, but not equal, and running more steps does not change that. Compare the two losses printed above: which line has the smaller loss, the learned one or the true one? Why does gradient descent not find the true line exactly?

*Your answer:* 

## ★★ Stretch: learning rates and scikit-learn

### Exercise 5 · What if the step is too big? (8 min)

The slide *What If the Step Is Too Big?* ran the same descent with three step sizes: $\eta = 0.05$, $0.45$ and $1.1$. **Guess first:** on our data, which one crawls, which one gets there fast, and which one diverges?

Then check. For each learning rate, run `fit_line` for 20 steps and store its list of losses in the dictionary `curves`, using the learning rate as the key. The next cell overlays the curves.

In [ ]:
curves = {}
for eta in [0.05, 0.45, 1.1]:
    # YOUR CODE HERE
    ...

In [ ]:
# Checkpoint
assert len(curves) >= 3 and all(len(c) == 20 for c in curves.values()), \
    "curves should map each learning rate to its list of 20 losses"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for eta, curve in curves.items():
    for ax in (ax1, ax2):
        ax.plot(curve, lw=2, marker="o", ms=3, label=f"$\\eta$ = {eta}")
start = next(iter(curves.values()))[0]
ax1.set(ylim=(0, 1.3 * start), xlabel="step", ylabel="loss", title="Zoomed in")
ax2.set(yscale="log", xlabel="step", ylabel="loss (log scale)", title="The whole story")
ax1.legend()
plt.show()

**Question.** Which learning rate diverges, and how do you see it in the curves? Now find the edge: edit the list of learning rates above with values between 0.45 and 1.1, and re-run both cells. Roughly what is the largest $\eta$ that still converges?

*Your answer:* 

### Exercise 6 · The same fit in scikit-learn (6 min)

The slide *Three Lines of scikit-learn*: every model speaks the same verbs, `fit`, `predict` and `score`. scikit-learn wants the inputs as a table `X` with one row per sample and one column per feature (the slide *Data as a Matrix*), so first we turn the 1-D array `x` into a table with a single column.

Create a `LinearRegression` model called `model` and fit it to `X` and `y` (two lines; `LinearRegression` was imported in the setup cell). Then fill in the verbs that predict and score.

In [ ]:
X = x.reshape(-1, 1)    # shape (N, 1): N rows, 1 feature
print("X has shape", X.shape)

# YOUR CODE HERE
...

y_pred = model.____(X)    # the learned line, at every x
print(f"scikit-learn:  w = {model.coef_[0]:.4f},  b = {model.intercept_:.4f}")
print(f"by hand:       w = {w:.4f},  b = {b:.4f}")
print(f"score (R^2) = {model.____(X, y):.3f}")

In [ ]:
# Checkpoint
assert np.isclose(model.coef_[0], w, atol=1e-3) and np.isclose(model.intercept_, b, atol=1e-3), \
    "The two fits should agree to about three decimals. Did Exercise 4 run with eta = 0.1 and 50 steps?"
assert np.allclose(y_pred, predict(x, w, b), atol=1e-2), "y_pred should be the model's predictions at X"
print("Same line, two ways.")

The score is $R^2$, the fraction of the variance of $y$ that the line explains: 1 is perfect, 0 is no better than predicting the mean. How to score a model *honestly* is tomorrow's topic.

**Question.** The two answers agree, yet scikit-learn did not run gradient descent: it never asked you for a learning rate. How did it find $w$ and $b$? (Look back at the slide *Linear Regression*.) And if that shortcut exists, why learn gradient descent at all?

*Your answer:* 

### Exercise 7 (optional) · Mini-batches (6 min)

The slide *Mini-Batches*: with millions of examples, computing the exact slope at every step is too slow, so we estimate it from a small random batch. The function below records every $(w, b)$ along the way, so that we can draw the path on the contour lines of the loss.

Fill in the loop body, three lines: pick `batch_size` random indices (`rng.choice(len(x), size=batch_size, replace=False)` does it), compute the gradient on those examples only with your `gradients` function, and take the step, as in Exercise 4.

In [ ]:
def descent_path(x, y, eta, n_steps, batch_size, w=-2.5, b=4.5):
    """Gradient descent on random batches. Returns the (w, b) at every step."""
    path = [(w, b)]
    for step in range(n_steps):
        # YOUR CODE HERE
        # HINT: x[idx] keeps only the entries of x listed in idx
        ...
        path.append((w, b))
    return np.array(path)


full = descent_path(x, y, eta=0.08, n_steps=40, batch_size=len(x))   # all the data, at every step
mini = descent_path(x, y, eta=0.08, n_steps=40, batch_size=4)        # four random examples per step

In [ ]:
# Checkpoint
assert np.allclose(full[-1], [w, b], atol=0.05), "With batch_size = len(x), the path should end at the minimum"
assert np.linalg.norm(mini[-1] - [w, b]) < 0.6, "The mini-batch path should end near the minimum"
print(f"all the data:  w = {full[-1, 0]:.2f},  b = {full[-1, 1]:.2f}")
print(f"mini-batches:  w = {mini[-1, 0]:.2f},  b = {mini[-1, 1]:.2f}")

w_axis, b_axis = np.linspace(-3.5, 4.5, 120), np.linspace(-1.5, 5.5, 120)
L_grid = np.array([[mse(y, predict(x, wi, bi)) for wi in w_axis] for bi in b_axis])   # your loss, at every (w, b)
W, B = np.meshgrid(w_axis, b_axis)
fig, ax = plt.subplots(figsize=(6, 5))
ax.contour(W, B, L_grid, levels=np.geomspace(L_grid.min() * 1.05, L_grid.max(), 14), colors=GREY, linewidths=0.6)
ax.plot(*full.T, "-o", color=INK, ms=3, lw=1.5, label=f"all the data ({len(x)} examples per step)")
ax.plot(*mini.T, "-o", color=LIME, ms=3, lw=1.5, label="mini-batches (4 examples per step)")
ax.plot(w, b, "*", color=MAGENTA, ms=16, label="the minimum")
ax.set(xlabel="$w$", ylabel="$b$", title="Two roads to the same valley")
ax.legend(fontsize=9, loc="lower left")
plt.show()

**Question.** Each mini-batch step looks at only 4 of the 100 points. Why does it still reach the valley? And why does it keep jittering near the bottom instead of settling?

*Your answer:* 

## ★★★ Challenge: logistic regression from scratch

Same recipe, different ingredients (slide *Logistic Regression as a Recipe*). The data are two classes of points in two dimensions, with labels $y_i \in \{0, 1\}$. The model squashes a linear score into a probability (slide *From Lines to Probabilities*):

$$p_i = \sigma(\mathbf w^\top \mathbf x_i + b), \qquad \sigma(z) = \frac{1}{1 + e^{-z}} .$$

The loss is the cross-entropy (slide *Cross-Entropy*), averaged over the data:

$$L(\mathbf w, b) = -\frac{1}{N}\sum_{i=1}^{N} \big[\,y_i \log p_i + (1 - y_i)\log(1 - p_i)\,\big] .$$

Now $\mathbf w$ has two entries, one per feature, and the inputs form a table `X2` with two columns. Run the cell below to make the data.

In [ ]:
clf_rng = np.random.default_rng(17)
X2 = np.vstack([clf_rng.normal([-1.1, -0.6], 0.9, size=(70, 2)),    # class 0 (cyan)
                clf_rng.normal([1.2, 0.9], 0.9, size=(70, 2))])     # class 1 (yellow)
y2 = np.r_[np.zeros(70), np.ones(70)]
print("X2 has shape", X2.shape, " y2 has shape", y2.shape)


def plot_classes(ax):
    """Scatter the two classes: cyan for 0, yellow for 1."""
    for label, color in [(0, CYAN), (1, YELLOW)]:
        ax.scatter(*X2[y2 == label].T, s=28, color=color, edgecolor="white", linewidth=0.5, label=f"class {label}")
    ax.set(xlabel="$x_1$", ylabel="$x_2$", xlim=(-4, 4), ylim=(-4, 4), aspect="equal")


fig, ax = plt.subplots(figsize=(5, 5))
plot_classes(ax)
ax.legend()
plt.show()

### Exercise 8 · Sigmoid, cross-entropy, gradient, update

Write the four pieces. The gradient of the cross-entropy is for you to work out (if you are stuck, there is a hint in the code). `fit_logistic` should record the loss at every step, like `fit_line`. The checkpoint below tests the pieces, then runs your `fit_logistic` for 1000 steps with $\eta = 1$.

In [ ]:
def sigmoid(z):
    """Squash any number into (0, 1)."""
    # YOUR CODE HERE
    ...


def cross_entropy(y, p):
    """The average cross-entropy between labels y (0 or 1) and probabilities p."""
    # YOUR CODE HERE
    ...


def logistic_gradients(X, y, w, b):
    """The gradient of the cross-entropy: (dL/dw, a vector with one entry per feature; dL/db)."""
    p = sigmoid(X @ w + b)    # the probability of class 1, for every row of X
    # YOUR CODE HERE
    # HINT: it looks like the straight line's gradient, with (p_i - y_i) in place of -2 r_i:
    # HINT: dL/dw = (1/N) sum_i (p_i - y_i) x_i   and   dL/db = (1/N) sum_i (p_i - y_i)
    ...
    return dL_dw, dL_db


def fit_logistic(X, y, eta, n_steps):
    """Gradient descent on the cross-entropy, from w = 0, b = 0. Returns w, b and the loss at every step."""
    w, b = np.zeros(X.shape[1]), 0.0
    losses = []
    for step in range(n_steps):
        # YOUR CODE HERE
        ...
    return w, b, losses

In [ ]:
# Checkpoint: first the pieces, then the whole fit
assert sigmoid(0.0) is not None and np.isclose(sigmoid(0.0), 0.5), "sigmoid(0) should be 1/2"
assert cross_entropy(np.array([1.0]), np.array([0.5])) is not None and \
    np.isclose(cross_entropy(np.array([1.0]), np.array([0.5])), np.log(2)), "a 50/50 guess should cost log 2 = 0.693"

w_lr, b_lr, ce_losses = fit_logistic(X2, y2, eta=1.0, n_steps=1000)
accuracy = np.mean((sigmoid(X2 @ w_lr + b_lr) > 0.5) == y2)
print(f"w = {np.round(w_lr, 3)},  b = {b_lr:.3f}")
print(f"cross-entropy: {ce_losses[0]:.3f} at the start, {ce_losses[-1]:.3f} at the end")
print(f"training accuracy: {accuracy:.0%}")
assert ce_losses[-1] < ce_losses[0], "the cross-entropy should go down"
assert accuracy > 0.9, "a straight boundary should classify more than 90% of these points correctly"
print("Logistic regression, from scratch.")

### Exercise 9 · The decision boundary, and scikit-learn

The model changes its mind where $p = 1/2$, that is, where $w_1 x_1 + w_2 x_2 + b = 0$ (slide *The Decision Boundary*). Solve this equation for $x_2$ to draw the boundary as a line, `x2_line`, over the points `x1_line`.

Then fit scikit-learn's `LogisticRegression` to `X2` and `y2` as `clf`. Use `LogisticRegression(C=np.inf)`: by default scikit-learn adds a penalty that keeps the weights small (more on penalties tomorrow), and `C=np.inf` switches it off, so that it minimises exactly your loss.

In [ ]:
# The boundary by hand: the x2 that solves w1 * x1 + w2 * x2 + b = 0, for each x1 in x1_line
x1_line = np.linspace(-4, 4, 2)
# YOUR CODE HERE
...

# The same model in scikit-learn, without penalty, fitted to X2 and y2
# YOUR CODE HERE
...
print(f"by hand:       w = {np.round(w_lr, 3)},  b = {b_lr:.3f}")
print(f"scikit-learn:  w = {np.round(clf.coef_[0], 3)},  b = {clf.intercept_[0]:.3f}")

In [ ]:
# Checkpoint
assert np.allclose(clf.coef_[0], w_lr, atol=0.01) and abs(clf.intercept_[0] - b_lr) < 0.01, \
    "With C=np.inf the two fits should agree to two decimals. If not, try more steps in fit_logistic."
assert abs(w_lr @ [x1_line[0], x2_line[0]] + b_lr) < 1e-9, "the points of the boundary should satisfy w1 x1 + w2 x2 + b = 0"

from matplotlib.colors import LinearSegmentedColormap
shade = LinearSegmentedColormap.from_list("cyan_to_yellow", ["#BFEFF6", "#FFFFFF", "#FFEFC2"])
g1, g2 = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-4, 4, 200))
grid = np.c_[g1.ravel(), g2.ravel()]
P = sigmoid(grid @ w_lr + b_lr).reshape(g1.shape)            # your probabilities
P_sk = clf.predict_proba(grid)[:, 1].reshape(g1.shape)       # scikit-learn's

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.contourf(g1, g2, P, levels=np.linspace(0, 1, 11), cmap=shade)
fig.colorbar(im, ax=ax, label="predicted probability of class 1 (yellow)")
plot_classes(ax)
ax.plot(x1_line, x2_line, color=INK, lw=2, label="boundary, by hand")
ax.contour(g1, g2, P_sk, levels=[0.5], colors=MAGENTA, linewidths=2, linestyles="--")
ax.plot([], [], color=MAGENTA, lw=2, ls="--", label="boundary, scikit-learn")   # legend entry for the contour
ax.set(title="Where the model changes its mind")
ax.legend(loc="lower right", fontsize=9)
plt.show()

### Going further (open-ended)

**Question.** Fit `LogisticRegression()` with its default settings and compare its weights and its boundary with yours. What changes, and what hardly changes?

*Your answer:* 

**Question.** Move the two clouds apart (for example, centres at $(-3, -3)$ and $(3, 3)$) so that a line separates them perfectly, and run your gradient descent for more and more steps. What happens to the loss, and to the size of $\mathbf w$?

*Your answer:* 

## Wrap-up

- **What you just did.** You wrote the whole recipe by hand: data, a model with two knobs, a loss, and gradient descent to turn the knobs. The learning rate was the first knob you had to tune yourself, and scikit-learn's `fit` did the same job in one line.
- **Where it comes back.** Every model this week is this recipe with different ingredients: the logistic regression of the Challenge is a single artificial neuron, and on Thursday, and then in Alessio's module, networks of such neurons are trained by the mini-batch gradient descent of Exercise 7.
- **Tomorrow.** Your line fits the training data, but will it work on new data? In Exercise 4 the learned line had a smaller loss than the true line, so a small loss on the data you trained on is not the whole story. Tuesday: *Not Fooling Yourself*.